In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_curve,
    auc
)
import matplotlib.pyplot as plt

# --- Load data ---
# Format: true_label, predicted_label, prob_class0, prob_class1, prob_class2, ...
data = np.loadtxt(r"C:\Users\Sam\Desktop\ML\data\Data_err.npt")
y_real = data[:, 0].astype(int)
y_pred_label = data[:, 1].astype(int)
y_pred_prob = data[:, 2:]  # predicted probabilities for each class

# --- Split into train/test ---
split_idx = int(len(y_real) * 0.8)
y_real_train, y_real_test = y_real[:split_idx], y_real[split_idx:]
y_pred_label_train, y_pred_label_test = y_pred_label[:split_idx], y_pred_label[split_idx:]
y_pred_prob_train, y_pred_prob_test = y_pred_prob[:split_idx], y_pred_prob[split_idx:]

# --- Metric function ---
def get_metrics(y_true, y_pred):
    return {
        "Recall": recall_score(y_true, y_pred, average='macro', zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, average='macro', zero_division=0),
        "Precision": precision_score(y_true, y_pred, average='macro', zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }

# --- Compute global metrics ---
metrics_all = get_metrics(y_real, y_pred_label)
metrics_train = get_metrics(y_real_train, y_pred_label_train)
metrics_test = get_metrics(y_real_test, y_pred_label_test)

df_main = pd.DataFrame([
    ["All", *metrics_all.values()],
    ["Train", *metrics_train.values()],
    ["Test", *metrics_test.values()],
], columns=["Set", "Recall", "Accuracy", "F1", "Precision", "MCC"])

# --- Per-class metrics ---
classes = np.unique(y_real)
precision_per_class = precision_score(y_real, y_pred_label, average=None, labels=classes, zero_division=0)
recall_per_class = recall_score(y_real, y_pred_label, average=None, labels=classes, zero_division=0)
f1_per_class = f1_score(y_real, y_pred_label, average=None, labels=classes, zero_division=0)

accuracy_per_class = []
class_error_per_class = []

for cls in classes:
    idx = y_real == cls
    acc = accuracy_score(y_real[idx], y_pred_label[idx])
    accuracy_per_class.append(acc)
    class_error_per_class.append(1 - acc)

df_class = pd.DataFrame({
    "Set": [f"Class {cls}" for cls in classes],
    "Recall": recall_per_class,
    "Accuracy": accuracy_per_class,
    "Class-Wise Error": class_error_per_class,
    "F1": f1_per_class,
    "Precision": precision_per_class,
    "MCC": ["" for _ in classes]  # Placeholder
})

# --- Combine both tables ---
df_combined = pd.concat([df_main, df_class], ignore_index=True)

# --- Confusion Matrix ---
cm = confusion_matrix(y_real, y_pred_label, labels=classes)
cm_df = pd.DataFrame(cm,
                     index=[f"Actual {cls}" for cls in classes],
                     columns=[f"Predicted {cls}" for cls in classes])

# --- Multi-class ROC and AUC ---
# Compute ROC per class 
roc_data = []
for i, cls in enumerate(classes):
    # True binary for this class
    y_true_bin = (y_real == cls).astype(int)
    y_score = y_pred_prob[:, i]
    
    fpr, tpr, thresholds = roc_curve(y_true_bin, y_score)
    roc_auc = auc(fpr, tpr)
    
    for j in range(len(fpr)):
        roc_data.append({
            "Class": cls,
            "FPR": fpr[j],
            "TPR": tpr[j],
            "Threshold": thresholds[j] if j < len(thresholds) else None,
            "AUC": roc_auc if j == len(fpr)-1 else ""
        })

roc_df = pd.DataFrame(roc_data)

# --- Display results ---
print("\n📊 Combined Metrics Table:")
print(df_combined.to_string(index=False))

print("\n🧮 Confusion Matrix:")
print(cm_df)

print("\n📈 ROC DataFrame (first few rows):")
# print(roc_df.head())

# Optional: copy metrics to clipboard
df_combined.to_clipboard(index=False)
# roc_df.to_clipboard(index=False)
cm_df.to_clipboard()



IndexError: index 0 is out of bounds for axis 1 with size 0

In [ ]:
# roc_df.to_clipboard(index=False)

In [30]:
cm_df.to_clipboard()